# LumenY — 02: Feature Engineering

Builds the full feature matrix from processed OHLCV data across all timeframes.

**Architecture:** 1H as base index, features computed from 5m, 15m, 1H, 4H, 1D timeframes and merged into one row per timestamp.

**Output:** One feature parquet per pair + one combined parquet for all pairs ready for model training.

In [ ]:
# !pip install pandas numpy pandas-ta scikit-learn pyarrow tqdm matplotlib

In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../backend/data/processed')
FEATURES_DIR  = Path('../backend/data/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']

print('Ready.')
print(f'Features will be saved to: {FEATURES_DIR.resolve()}')

## 1. Feature Engineering Functions

Each function takes a OHLCV DataFrame and returns a DataFrame of features with a `_{tf}` suffix on column names so we know which timeframe each feature came from.

In [ ]:
def compute_features(df: pd.DataFrame, tf: str) -> pd.DataFrame:
    """
    Compute all TA features for a given OHLCV DataFrame.
    Returns a DataFrame with all features, columns suffixed with _{tf}.
    All features use only past data — no lookahead.
    """
    feat = pd.DataFrame(index=df.index)
    o, h, l, c = df['open'], df['high'], df['low'], df['close']
    
    # ── RETURNS & MOMENTUM ──────────────────────────────────────────
    
    # Log returns at multiple lookbacks
    for n in [1, 3, 6, 12, 24, 48]:
        feat[f'log_ret_{n}'] = np.log(c / c.shift(n))
    
    # RSI
    feat['rsi_14'] = ta.rsi(c, length=14)
    feat['rsi_28'] = ta.rsi(c, length=28)
    
    # RSI slope — is momentum accelerating or decelerating?
    feat['rsi_slope'] = feat['rsi_14'] - feat['rsi_14'].shift(3)
    
    # RSI divergence flag — price makes new high but RSI doesn't (or vice versa)
    price_higher = (c > c.shift(5)).astype(int)
    rsi_higher   = (feat['rsi_14'] > feat['rsi_14'].shift(5)).astype(int)
    feat['rsi_divergence'] = (price_higher != rsi_higher).astype(int)
    
    # MACD
    macd = ta.macd(c, fast=12, slow=26, signal=9)
    if macd is not None:
        feat['macd']        = macd.iloc[:, 0]
        feat['macd_signal'] = macd.iloc[:, 2]
        feat['macd_hist']   = macd.iloc[:, 1]
    
    # Distance from moving averages (normalized by price)
    for n in [20, 50, 200]:
        ma = ta.sma(c, length=n)
        feat[f'dist_ma_{n}'] = (c - ma) / c
    
    # EMA cross signal
    ema_fast = ta.ema(c, length=9)
    ema_slow = ta.ema(c, length=21)
    feat['ema_cross'] = (ema_fast > ema_slow).astype(int)
    feat['ema_dist']  = (ema_fast - ema_slow) / c

    # ── VOLATILITY ──────────────────────────────────────────────────
    
    # ATR (normalized by price)
    atr_14 = ta.atr(h, l, c, length=14)
    atr_28 = ta.atr(h, l, c, length=28)
    feat['atr_14_norm'] = atr_14 / c
    feat['atr_28_norm'] = atr_28 / c
    
    # ATR ratio — current vs longer-term (compression/expansion)
    feat['atr_ratio'] = atr_14 / atr_28
    
    # Rolling std of log returns
    log_ret = np.log(c / c.shift(1))
    feat['rvol_12']  = log_ret.rolling(12).std()
    feat['rvol_24']  = log_ret.rolling(24).std()
    feat['rvol_48']  = log_ret.rolling(48).std()
    
    # Realized vol ratio — short vs long (are we in a vol spike?)
    feat['rvol_ratio'] = feat['rvol_12'] / feat['rvol_48']
    
    # High-low range normalized
    feat['hl_range'] = (h - l) / c
    
    # Bollinger Band width and position
    bb = ta.bbands(c, length=20, std=2)
    if bb is not None:
        bb_upper = bb.iloc[:, 0]
        bb_mid   = bb.iloc[:, 1]
        bb_lower = bb.iloc[:, 2]
        feat['bb_width']    = (bb_upper - bb_lower) / bb_mid
        feat['bb_position'] = (c - bb_lower) / (bb_upper - bb_lower + 1e-10)
    
    # ── MARKET STRUCTURE & KEY LEVELS ───────────────────────────────
    
    # Distance from N-bar high/low (normalized)
    for n in [20, 50]:
        feat[f'dist_high_{n}'] = (c - h.rolling(n).max()) / c
        feat[f'dist_low_{n}']  = (c - l.rolling(n).min()) / c
    
    # Breakout flag — price above N-bar high
    feat['breakout_20'] = (c > h.shift(1).rolling(20).max()).astype(int)
    feat['breakdown_20'] = (c < l.shift(1).rolling(20).min()).astype(int)
    
    # Trend slope — linear regression slope of close over N bars
    def rolling_slope(series, n):
        slopes = series.copy() * np.nan
        x = np.arange(n)
        for i in range(n, len(series)):
            y = series.iloc[i-n:i].values
            if not np.any(np.isnan(y)):
                slopes.iloc[i] = np.polyfit(x, y, 1)[0] / series.iloc[i]
        return slopes
    
    feat['trend_slope_20'] = rolling_slope(c, 20)
    
    # Candle body and wick ratios
    body  = abs(c - o)
    range_ = h - l + 1e-10
    feat['body_ratio']       = body / range_
    feat['upper_wick_ratio'] = (h - pd.concat([c, o], axis=1).max(axis=1)) / range_
    feat['lower_wick_ratio'] = (pd.concat([c, o], axis=1).min(axis=1) - l) / range_
    
    # ADX — trend strength
    adx = ta.adx(h, l, c, length=14)
    if adx is not None:
        feat['adx'] = adx.iloc[:, 0]
    
    # Add timeframe suffix to all columns
    feat.columns = [f'{col}_{tf}' for col in feat.columns]
    
    return feat


print('Feature function ready.')

In [ ]:
def compute_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute time-based features from the index.
    These are the same regardless of timeframe.
    """
    feat = pd.DataFrame(index=df.index)
    
    # Hour of day — FX session patterns
    feat['hour'] = df.index.hour
    
    # Session flags
    feat['session_asian']  = ((df.index.hour >= 0)  & (df.index.hour < 8)).astype(int)
    feat['session_london'] = ((df.index.hour >= 8)  & (df.index.hour < 16)).astype(int)
    feat['session_ny']     = ((df.index.hour >= 13) & (df.index.hour < 21)).astype(int)
    feat['session_overlap'] = ((df.index.hour >= 13) & (df.index.hour < 16)).astype(int)  # London/NY overlap — most volatile
    
    # Day of week
    feat['day_of_week'] = df.index.dayofweek  # 0=Mon, 4=Fri
    feat['is_monday']   = (df.index.dayofweek == 0).astype(int)
    feat['is_friday']   = (df.index.dayofweek == 4).astype(int)
    
    # Month — some FX seasonality exists
    feat['month'] = df.index.month
    
    return feat


print('Time feature function ready.')

In [ ]:
def compute_crossTF_features(feat_1h: pd.DataFrame, feat_4h: pd.DataFrame, feat_1d: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cross-timeframe alignment features.
    These capture confluence — when multiple timeframes agree on direction.
    """
    feat = pd.DataFrame(index=feat_1h.index)
    
    # Trend alignment — are 1H and 4H moving averages in same direction?
    if 'ema_cross_1H' in feat_1h.columns and 'ema_cross_4H' in feat_4h.columns:
        feat['trend_align_1h_4h'] = (feat_1h['ema_cross_1H'] == feat_4h['ema_cross_4H']).astype(int)
    
    if 'ema_cross_4H' in feat_4h.columns and 'ema_cross_1D' in feat_1d.columns:
        feat['trend_align_4h_1d'] = (feat_4h['ema_cross_4H'] == feat_1d['ema_cross_1D']).astype(int)
    
    # Full confluence — all three timeframes agree
    if all(c in feat.columns for c in ['trend_align_1h_4h', 'trend_align_4h_1d']):
        feat['full_confluence'] = (feat['trend_align_1h_4h'] & feat['trend_align_4h_1d']).astype(int)
    
    # Volatility regime — is short-term vol expanding vs long-term?
    if 'atr_ratio_1H' in feat_1h.columns and 'atr_ratio_1D' in feat_1d.columns:
        feat['vol_expansion'] = (feat_1h['atr_ratio_1H'] > feat_1d['atr_ratio_1D']).astype(int)
    
    # RSI alignment across timeframes
    if 'rsi_14_1H' in feat_1h.columns and 'rsi_14_4H' in feat_4h.columns:
        feat['rsi_align_1h_4h'] = np.sign(feat_1h['rsi_14_1H'] - 50) == np.sign(feat_4h['rsi_14_4H'] - 50)
        feat['rsi_align_1h_4h'] = feat['rsi_align_1h_4h'].astype(int)
    
    # Momentum confluence score — how many timeframes show bullish momentum?
    scores = []
    for col, df_ in [('rsi_14_1H', feat_1h), ('rsi_14_4H', feat_4h), ('rsi_14_1D', feat_1d)]:
        if col in df_.columns:
            scores.append((df_[col] > 50).astype(int))
    if scores:
        feat['momentum_confluence'] = sum(scores)
    
    return feat


print('Cross-TF feature function ready.')

## 2. Build Feature Matrix Per Pair

In [ ]:
def build_feature_matrix(pair: str) -> pd.DataFrame:
    """
    Build the full feature matrix for a given pair.
    Base index = 1H candles.
    Features from 5m, 15m, 1H, 4H, 1D — all aligned to 1H timestamps.
    """
    print(f'  Loading data...')
    
    # Load all timeframes
    dfs = {}
    for tf in ['5m', '15m', '1H', '4H', '1D']:
        path = PROCESSED_DIR / f'{pair}_{tf}.parquet'
        if path.exists():
            dfs[tf] = pd.read_parquet(path)
        else:
            print(f'  WARNING: Missing {tf} data for {pair}')
    
    # Base index is 1H
    base = dfs['1H'].copy()
    
    print(f'  Computing features...')
    
    # Compute features for each timeframe
    feat_5m  = compute_features(dfs['5m'],  '5m')  if '5m'  in dfs else None
    feat_15m = compute_features(dfs['15m'], '15m') if '15m' in dfs else None
    feat_1h  = compute_features(dfs['1H'],  '1H')
    feat_4h  = compute_features(dfs['4H'],  '4H')  if '4H'  in dfs else None
    feat_1d  = compute_features(dfs['1D'],  '1D')  if '1D'  in dfs else None
    
    # Time features (based on 1H index)
    feat_time = compute_time_features(base)
    
    print(f'  Aligning timeframes to 1H index...')
    
    # Start with 1H features (same index)
    all_features = feat_1h.copy()
    
    # Merge 5m features — resample to 1H using last value (no lookahead)
    if feat_5m is not None:
        feat_5m_1h = feat_5m.resample('1h').last()
        all_features = all_features.join(feat_5m_1h, how='left')
    
    # Merge 15m features — resample to 1H
    if feat_15m is not None:
        feat_15m_1h = feat_15m.resample('1h').last()
        all_features = all_features.join(feat_15m_1h, how='left')
    
    # Merge 4H features — forward fill to 1H (each 4H value persists for 4 hours)
    if feat_4h is not None:
        feat_4h_1h = feat_4h.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_4h_1h, how='left')
    
    # Merge 1D features — forward fill to 1H
    if feat_1d is not None:
        feat_1d_1h = feat_1d.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_1d_1h, how='left')
    
    # Cross-TF features
    if feat_4h is not None and feat_1d is not None:
        feat_4h_aligned = feat_4h.reindex(all_features.index, method='ffill')
        feat_1d_aligned = feat_1d.reindex(all_features.index, method='ffill')
        feat_cross = compute_crossTF_features(feat_1h, feat_4h_aligned, feat_1d_aligned)
        all_features = all_features.join(feat_cross, how='left')
    
    # Time features
    all_features = all_features.join(feat_time, how='left')
    
    # Pair identity — encoded as integer
    pair_map = {p: i for i, p in enumerate(PAIRS)}
    all_features['pair_id'] = pair_map[pair]
    
    print(f'  Feature matrix shape: {all_features.shape}')
    
    return all_features


print('Feature matrix builder ready.')

In [ ]:
# Build and save feature matrix for each pair
for pair in PAIRS:
    print(f'\nBuilding features for {pair}...')
    
    try:
        feat_df = build_feature_matrix(pair)
        out_path = FEATURES_DIR / f'{pair}_features.parquet'
        feat_df.to_parquet(out_path)
        print(f'  Saved: {out_path.name} — {feat_df.shape[0]} rows × {feat_df.shape[1]} features')
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()

print('\nAll pairs done!')

## 2b. Cross-Pair Correlation Features

In [ ]:
# ## 2b. Cross-Pair Correlation Features
# Rolling correlation between each pair and all others.
# Captures regime shifts — during geopolitical stress or risk-off events,
# normally uncorrelated pairs suddenly move together.

print('Computing cross-pair correlations...')

# Load 1H close prices for all pairs
closes = {}
for pair in PAIRS:
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    closes[pair] = df_1h['close']

close_df = pd.DataFrame(closes)

# Compute rolling correlations at two windows
# 24H = short-term correlation regime
# 168H = weekly correlation regime
for window, w_name in [(24, '24H'), (168, '1W')]:
    print(f'  Computing {w_name} rolling correlations...')
    rolling_corr = close_df.rolling(window).corr()
    
    for pair in PAIRS:
        feat_path = FEATURES_DIR / f'{pair}_features.parquet'
        pair_data = pd.read_parquet(feat_path)
        
        for other_pair in PAIRS:
            if other_pair == pair:
                continue
            col_name = f'corr_{other_pair}_{w_name}'
            try:
                corr_series = rolling_corr.xs(pair, level=1)[other_pair]
                pair_data[col_name] = corr_series.reindex(pair_data.index, method='ffill')
            except Exception as e:
                pair_data[col_name] = np.nan
        
        pair_data.to_parquet(feat_path)
    
    print(f'  {w_name} correlations added to all pairs.')

# Verify
sample = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
corr_cols = [c for c in sample.columns if c.startswith('corr_')]
print(f'\nCorrelation features added: {len(corr_cols)}')
print(f'Example columns: {corr_cols[:4]}')
print('Cross-pair correlations complete.')


## 3. Compute Labels

For each horizon we compute the forward log return.
This is the target variable `μ` our regression models will predict.

**No lookahead:** labels use future candles only. Features use past candles only.

In [ ]:
# Horizon definitions in number of 1H bars
HORIZONS = {
    '1H':  1,
    '4H':  4,
    '1D':  24,
    '7D':  168,
}

def compute_labels(pair: str, feat_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute forward log returns for each horizon.
    Uses 1H close prices.
    """
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    close = df_1h['close'].reindex(feat_df.index)
    
    labels = pd.DataFrame(index=feat_df.index)
    
    for horizon_name, n_bars in HORIZONS.items():
        # Forward log return — shift(-n) gives us future price
        labels[f'label_{horizon_name}'] = np.log(close.shift(-n_bars) / close)
    
    return labels


print('Label function ready.')
print(f'Horizons: {HORIZONS}')

In [ ]:
# Combine all pairs into one dataset with features + labels
all_dfs = []

for pair in PAIRS:
    print(f'Processing labels for {pair}...')
    
    feat_path = FEATURES_DIR / f'{pair}_features.parquet'
    if not feat_path.exists():
        print(f'  Missing features, skipping.')
        continue
    
    feat_df = pd.read_parquet(feat_path)
    label_df = compute_labels(pair, feat_df)
    
    combined = feat_df.join(label_df, how='left')
    combined['pair'] = pair
    
    all_dfs.append(combined)
    print(f'  {pair}: {combined.shape}')

# Combine all pairs
df_all = pd.concat(all_dfs, axis=0)
df_all = df_all.sort_index()

print(f'\nCombined dataset shape: {df_all.shape}')
print(f'Date range: {df_all.index[0]} -> {df_all.index[-1]}')
print(f'Pairs: {df_all["pair"].unique()}')

In [ ]:
# Drop rows where any label is NaN (end of dataset — no future data)
label_cols = [f'label_{h}' for h in HORIZONS.keys()]
df_all = df_all.dropna(subset=label_cols)

# Save combined dataset
out_path = FEATURES_DIR / 'all_pairs_features_labels.parquet'
df_all.to_parquet(out_path)

print(f'Combined dataset saved: {df_all.shape}')
print(f'Features: {df_all.shape[1] - len(label_cols) - 1} columns')
print(f'Labels: {label_cols}')
print(f'\nLabel statistics:')
df_all[label_cols].describe()

## 4. Validate Features

In [ ]:
# Check for NaN ratios per feature — high NaN = feature may be problematic
feat_cols = [c for c in df_all.columns if c not in label_cols + ['pair']]

nan_ratio = df_all[feat_cols].isnull().mean().sort_values(ascending=False)
print('Features with >10% NaN:')
print(nan_ratio[nan_ratio > 0.1])
print(f'\nTotal features: {len(feat_cols)}')
print(f'Features with 0% NaN: {(nan_ratio == 0).sum()}')

In [ ]:
# Label distribution — check for class balance and outliers
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#080c14')

for ax, horizon in zip(axes.flatten(), HORIZONS.keys()):
    col = f'label_{horizon}'
    data = df_all[col].dropna()
    
    # Clip extreme outliers for visualization
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    
    ax.hist(data_clipped, bins=100, color='#4fc3f7', alpha=0.7, edgecolor='none')
    ax.axvline(0, color='#ff4757', linewidth=1.5, linestyle='--')
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    ax.set_title(f'{horizon} Returns', color='white')
    
    pct_up = (data > 0).mean()
    ax.text(0.02, 0.95, f'Up: {pct_up:.1%}  Down: {1-pct_up:.1%}',
            transform=ax.transAxes, color='white', fontsize=9, va='top')
    
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

plt.suptitle('Label Distributions by Horizon', color='white', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation check — flag highly correlated features (>0.95)
# High correlation = redundant features that add noise
sample = df_all[feat_cols].dropna().sample(min(10000, len(df_all)))
corr_matrix = sample.corr().abs()

# Find highly correlated pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[row, col]) 
             for col in upper.columns 
             for row in upper.index 
             if upper.loc[row, col] > 0.95]

print(f'Highly correlated feature pairs (>0.95): {len(high_corr)}')
for a, b, v in sorted(high_corr, key=lambda x: -x[2])[:20]:
    print(f'  {a} <-> {b}: {v:.3f}')

In [ ]:
# Final summary
print('=' * 50)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 50)
print(f'Total rows:     {len(df_all):,}')
print(f'Total features: {len(feat_cols)}')
print(f'Pairs:          {len(PAIRS)}')
print(f'Date range:     {df_all.index[0].date()} -> {df_all.index[-1].date()}')
print(f'\nLabel columns:  {label_cols}')
print(f'\nOutput file:    all_pairs_features_labels.parquet')